# TradeSense · Component 03 · Autoencoder improvement cycle → `models_autoencoder_v2`
**One targeted cycle, not open-ended tuning.** Autoencoder v1 trained correctly (15/15) but detected few synthetic events, and the 15-bar model detected almost none at its q99.5 threshold. This notebook (1) diagnoses *why*, (2) compares causal scoring rules on the existing v1 models, (3) trains a small, justified set of new architectures only if scoring is insufficient, and (4) compares everything at a **matched real-market alarm burden** (alarm episodes per day on genuine Layer 1 validation data).

**Rules baked in (frozen in a hashed protocol before any score is computed):**
* All fitting and preprocessing: genuine Layer 1 **TRAIN** only. Thresholds: Layer 1 **VALIDATION** only.
* **Layer 3 labels never fit or select a model.** Development selection uses a *Layer 1 validation pseudo-anomaly probe* (generic short feature-space perturbations of genuine validation bars, not the Layer 3 generator) plus label-free burden-stability gates.
* Layer 3 **validation** is used for the descriptive diagnosis and for one pre-registered go/no-go comparison against the v1 single-bar baseline (fallback rule). Layer 3 **test** was already examined in v1: it is evaluated **once**, only for a pre-declared handful of models, and labelled *exploratory, not independent confirmation*.
* Nothing in `models_autoencoder_v1`, any dataset, or any notebook is modified.

## 0 · Setup and frozen-input pinning

In [1]:
import os, sys, json, time, math, re, hashlib, shutil, uuid, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import stats
from IPython.display import display
import torch, torch.nn as nn

warnings.filterwarnings('ignore', category=FutureWarning); warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 120); pd.set_option('display.float_format', lambda v: f'{v:,.4g}')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L1 = BASE / 'processed_layer1_final_v1'; L3 = BASE / 'processed_layer3_final_v2'; V1 = BASE / 'models_autoencoder_v1'
OUT_FINAL = BASE / 'models_autoencoder_v2'; STAGE = BASE / ('.ae2_stage_' + uuid.uuid4().hex[:8])
FROZEN_NAMES = ['processed_layer1_final_v1', 'processed_layer2_final_v3', 'processed_layer2_model_ready_v1', 'processed_layer3_final_v1', 'processed_layer3_final_v2', 'models_autoencoder_v1']
for p in (L1, L3, V1): assert p.is_dir(), f'required input missing: {p}'
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); TS = {}
print('base:', BASE, '| device:', DEVICE, '| torch', torch.__version__)

def sha256_file(p, chunk=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def pin_dir(d): return {str(p.relative_to(d)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)} for p in sorted(d.rglob('*')) if p.is_file()}
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def auc_rank(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]; n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0 or np.nanstd(x) == 0: return float('nan')
    r = stats.rankdata(x); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def avg_precision(x, y):
    x = np.asarray(x, float); y = np.asarray(y).astype(int); m = np.isfinite(x); x, y = x[m], y[m]
    if y.sum() == 0: return float('nan')
    o = np.argsort(-x, kind='mergesort'); yy = y[o]; tp = np.cumsum(yy); return float(((tp / np.arange(1, len(yy) + 1)) * yy).sum() / yy.sum())
def wilson(k, n, z=1.96):
    if n == 0: return [float('nan'), float('nan')]
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d; return [max(0.0, c - h), min(1.0, c + h)]
t0 = time.time(); PIN = {n: pin_dir(BASE / n) for n in FROZEN_NAMES if (BASE / n).is_dir()}
print('pinned:', {n: len(v) for n, v in PIN.items()}, f'({time.time()-t0:.0f}s)')

Mounted at /content/drive
base: /content/drive/MyDrive/research datasets | device: cuda | torch 2.11.0+cu130
pinned: {'processed_layer1_final_v1': 400, 'processed_layer2_final_v3': 8, 'processed_layer2_model_ready_v1': 27, 'processed_layer3_final_v1': 5, 'processed_layer3_final_v2': 32, 'models_autoencoder_v1': 39} (182s)


## 1 · FREEZE THE v2 PROTOCOL (before any v2 score, probe or model exists)

In [2]:
FEATURES = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
SYMS = {'BTCUSDT': 'btc', 'ETHUSDT': 'eth'}; SPL = ('train', 'val', 'test'); WARM = 15; N_BARS = 129600
BOUNDS = {'train': (0, 90720), 'val': (90720, 109440), 'test': (109440, 129600)}; SEEDS = [11, 22, 33]
PROTOCOL = {
 'protocol_id': 'ae_improvement_protocol_v2', 'parent': 'ae_evaluation_protocol_v1 (sha256 recorded in models_autoencoder_v1/protocol)', 'statement': 'Written and hashed before any v2 score, probe or model exists.',
 'burden_definition': 'alarm EPISODES per day on genuine Layer 1 VALIDATION bars (alarm bars merged when gap <= 5 bars). Thresholds are chosen per model so that this equals the budget (largest alarm set with episodes/day <= budget).',
 'budgets_episodes_per_day': [1, 2, 4], 'primary_budget': 2,
 'scoring_methods': {'window15 (existing v1 models, no retraining)': 'temporal aggregator over the 15 per-bar reconstruction errors of the window ending at t: mean (v1 rule), max, q90, current-bar; each with feature aggregator fmean (mean of the 6 feature errors) or fmax (max over features of error / feature scale, scale = TRAIN 99th percentile of that feature error)',
                     'single_bar (existing v1 models)': 'current-bar error with fmean or fmax'},
 'architecture_stage': {'run_if': 'best window15 scoring method does NOT reach the v1 single-bar probe recall at the primary budget (sufficiency gate)', 'candidates': {
     'ctx_bar': 'conditional autoencoder: encoder input = current bar (6) + causal context (mean and std of the PREVIOUS 15 bars, 12); decoder sees latent(3) + context; target = current bar only; score = current-bar error (fmean, fmax)',
     'dn_bar': 'single-bar denoising autoencoder (Gaussian input noise sd 0.15 in scaled space during training); score = current-bar error (fmean, fmax)'}, 'seeds': SEEDS,
     'training': 'Layer 1 TRAIN only; fit = first 90% of valid train windows, early stopping = last 10% (15-bar gap); Adam 1e-3, wd 1e-5, batch 512, max 150 epochs, patience 10, MSE'},
 'development_probe': {'definition': 'Layer 1 VALIDATION pseudo-anomaly probe: 117 events (9 strata = 3 durations {1-3, 4-12, 13-40 bars} x 3 magnitudes {2, 4, 8 robust-SD units}, 13 events each), each shifts 1-3 randomly chosen features of the scaled series by +/- magnitude x half-sine envelope; events >= 60 bars apart; negatives = bars outside [start, end+15]. Generic, not the Layer 3 generator; no Layer 3 label touched.',
                       'seed': 20261012, 'metric': 'event recall at the primary budget, macro-averaged over the 9 strata, averaged over BTC and ETH', 'honest_caveat': 'feature-space perturbations lack cross-feature coherence; the probe ranks sensitivity, it is not evidence of manipulation detection.'},
 'selection_rule': {'step1_gate': 'exclude a candidate if its budget-2 threshold yields > 3x the budget in episodes/day on the genuine hold-out tail of TRAIN (burden does not transfer)', 'step2': 'maximise probe score; candidates within 0.02 of the best are tied', 'step3_tie_break': 'prefer the simpler family: single-bar < ctx_bar/dn_bar < window15',
                    'step4_same_rank': 'within the same simplicity rank choose the highest probe score, then alphabetical scorer name', 'tail_definition': 'genuine Layer 1 TRAIN bars after the early-stopping boundary (last 10% of TRAIN, 15-bar gap)', 'labels_used': 'none from Layer 3'},
 'fallback_rule': 'After selection, if the selected model\'s Layer 3 VALIDATION event recall at the primary budget (mean of BTC, ETH) is lower than the v1 single-bar model\'s at the same budget, the recommendation is the v1 single-bar model (reported as a fallback, not as a selection).',
 'layer3_use': {'diagnosis': 'descriptive, VALIDATION only', 'final_report': 'selected model + pre-declared comparators only: v1 window15 (mean), v1 single_bar (fmean), robust-z baseline', 'test': 'already examined in v1 -> evaluated once, exploratory, not independent confirmation; no other candidate is evaluated on test'},
 'bootstrap': {'B': 200, 'seed': 7}, 'claims_policy': 'Scores are anomaly signals; no claim of manipulation detection. Layer 3 labels describe injection mechanisms.'}
PJ = json.dumps(PROTOCOL, sort_keys=True, separators=(',', ':'), ensure_ascii=False); PROTOCOL_HASH = hashlib.sha256(PJ.encode()).hexdigest()
STAGE.mkdir(parents=True, exist_ok=False)
for sub in ('protocol', 'models', 'scores', 'reports', 'validation'): (STAGE / sub).mkdir()
dump_json({**PROTOCOL, 'integrity': {'sha256_of_canonical_json_excluding_this_field': PROTOCOL_HASH, 'frozen_utc': UTC()}}, STAGE / 'protocol' / 'improvement_protocol.json')
TS['protocol_frozen'] = UTC(); print('PROTOCOL v2 FROZEN  sha256 =', PROTOCOL_HASH)

PROTOCOL v2 FROZEN  sha256 = 7e3d340080d00a5eb8de64f23c65443e15b767997d3812c464dffeecb40dffde


## 2 · Load genuine Layer 1 (fit/threshold), Layer 3 v2 (kept locked until selection), and the v1 models; verify v1 is reproduced exactly

In [3]:
F1, DT, F3, LB3 = {}, {}, {}, {}
for sym, tag in SYMS.items():
    mp = L1 / 'processed_matrices' / f'{sym.lower()}_l1_feature_matrix_90d.parquet'
    d = pd.read_parquet(mp); d['datetime'] = pd.to_datetime(d['datetime'], utc=True); d = d.sort_values('datetime').reset_index(drop=True)
    assert len(d) == N_BARS and all(f in d.columns for f in FEATURES); F1[sym] = d[FEATURES].to_numpy(np.float64); DT[sym] = d['datetime']
    fe = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_features.parquet') for sp in SPL], ignore_index=True); lb = pd.concat([pd.read_parquet(L3 / tag / f'{sp}_labels.parquet') for sp in SPL], ignore_index=True)
    assert list(fe.columns) == ['datetime'] + FEATURES and len(fe) == N_BARS and (pd.to_datetime(fe.datetime, utc=True).reset_index(drop=True) == DT[sym]).all() and (pd.to_datetime(lb.datetime, utc=True).reset_index(drop=True) == DT[sym]).all()
    F3[sym] = fe[FEATURES].to_numpy(np.float64); LB3[sym] = lb
EV3 = pd.concat([pd.read_parquet(L3 / 'events' / f'{t}_events.parquet') for t in SYMS.values()], ignore_index=True)

def transform(F):
    X = np.asarray(F, np.float64).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); return X
PREP, Z1, Z3 = {}, {}, {}
for sym in SYMS:
    X = transform(F1[sym]); tr = X[WARM:BOUNDS['train'][1]]
    med = np.median(tr, axis=0); q75, q25 = np.percentile(tr, [75, 25], axis=0); sc = (q75 - q25) / 1.349
    sc = np.where(sc > 1e-12, sc, np.maximum(tr.std(axis=0), 1e-8)); sc = np.maximum(sc, 1e-8)
    PREP[sym] = {'median': med, 'scale': sc, 'fit_rows': [WARM, BOUNDS['train'][1]], 'source': 'processed_layer1_final_v1 TRAIN only'}
    Z1[sym] = np.clip((X - med) / sc, -20, 20).astype(np.float32); Z3[sym] = np.clip((transform(F3[sym]) - med) / sc, -20, 20).astype(np.float32)
    v1p = json.loads((V1 / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json').read_text())
    assert np.allclose(v1p['median'], med, rtol=1e-9, atol=1e-12) and np.allclose(v1p['scale'], sc, rtol=1e-9, atol=1e-12), 'preprocessing differs from v1'
V1PROT = json.loads((V1 / 'protocol' / 'evaluation_protocol.json').read_text()); V1_HASH = V1PROT['integrity']['sha256_of_canonical_json_excluding_this_field']
V1THR = json.loads((V1 / 'models' / 'thresholds.json').read_text())['thresholds']
print('Layer 1 / Layer 3 v2 loaded; preprocessing identical to v1; v1 protocol sha256', V1_HASH[:16], '| Layer 3 events', len(EV3))
TS['data_loaded'] = UTC()

Layer 1 / Layer 3 v2 loaded; preprocessing identical to v1; v1 protocol sha256 84803e81384e3d40 | Layer 3 events 292


## 3 · Models, per-position error tensors and the scoring registry

For a 15-bar window the autoencoder reconstructs 15 x 6 values. v1 scored the window with the **mean** of those 90 squared errors. Here every position keeps its own error `E[t, k, f]` (bar k of the window ending at t, feature f), so temporal and feature aggregation can be compared causally. Score at bar t uses only bars `t-14..t`.

In [4]:
from numpy.lib.stride_tricks import sliding_window_view
def windows(Z, W): return sliding_window_view(Z, (W, Z.shape[1]))[:, 0].reshape(len(Z) - W + 1, W * Z.shape[1])
class AE(nn.Module):                      # identical to v1
    def __init__(s, d, b):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(d, 64), nn.LeakyReLU(0.1), nn.Linear(64, 32), nn.LeakyReLU(0.1), nn.Linear(32, b))
        s.dec = nn.Sequential(nn.Linear(b, 32), nn.LeakyReLU(0.1), nn.Linear(32, 64), nn.LeakyReLU(0.1), nn.Linear(64, d))
    def forward(s, x): return s.dec(s.enc(x))
class CAE(nn.Module):                     # d_ctx = 12: conditional AE (ctx_bar); d_ctx = 0: plain single-bar AE (dn_bar)
    def __init__(s, d_ctx, b=3):
        super().__init__(); s.d_ctx = d_ctx
        s.enc = nn.Sequential(nn.Linear(6 + d_ctx, 64), nn.LeakyReLU(0.1), nn.Linear(64, 32), nn.LeakyReLU(0.1), nn.Linear(32, b))
        s.dec = nn.Sequential(nn.Linear(b + d_ctx, 32), nn.LeakyReLU(0.1), nn.Linear(32, 64), nn.LeakyReLU(0.1), nn.Linear(64, 6))
    def forward(s, x, c=None):
        if s.d_ctx == 0: return s.dec(s.enc(x))
        return s.dec(torch.cat([s.enc(torch.cat([x, c], 1)), c], 1))
def ctx_feats(Z):                         # causal context for bar t: mean and std (ddof 0) of bars t-15..t-1
    n = len(Z); C = np.full((n, 12), np.nan, np.float32); win = sliding_window_view(Z.astype(np.float64), (15, 6))[:, 0]
    C[15:] = np.concatenate([win[:n - 15].mean(axis=1), win[:n - 15].std(axis=1)], axis=1).astype(np.float32); return C
FIRST_VALID = {'w15': WARM + 14, 'sb': WARM, 'dn': WARM, 'ctx': WARM + 15}           # first bar with a legal input (no masked bar inside the input or context)

V1M = {}
for sym, tag in SYMS.items():
    for var, fam in (('window15', 'w15'), ('single_bar', 'sb')):
        arch = json.loads((V1 / 'models' / tag / var / 'arch.json').read_text()); ms = []
        for sd in SEEDS:
            m = AE(arch['d'], arch['bottleneck']); m.load_state_dict(torch.load(V1 / 'models' / tag / var / f'seed{sd}.pt', map_location='cpu')); ms.append(m.to(DEVICE).eval())
        V1M[(sym, fam)] = ms
NEWM = {}                                  # (sym, fam) -> list of CAE, filled by the architecture stage

@torch.no_grad()
def errs(sym, fam, Z):
    """Per-position squared errors, ensemble mean over seeds. w15: (n,15,6); others: (n,6). NaN before the first legal bar."""
    ms = V1M[(sym, fam)] if fam in ('w15', 'sb') else NEWM[(sym, fam)]; n = len(Z)
    if fam == 'w15':
        Wn = windows(Z, 15); out = np.zeros((len(Wn), 15, 6), np.float32)
        for i in range(0, len(Wn), 8192):
            x = torch.tensor(Wn[i:i + 8192], device=DEVICE); acc = 0
            for m in ms: acc = acc + ((m(x) - x) ** 2).reshape(len(x), 15, 6)
            out[i:i + 8192] = (acc / len(ms)).cpu().numpy()
        full = np.full((n, 15, 6), np.nan, np.float32); full[14:] = out
    else:
        C = ctx_feats(Z) if fam == 'ctx' else None; out = np.zeros((n, 6), np.float32)
        for i in range(0, n, 8192):
            x = torch.tensor(Z[i:i + 8192], device=DEVICE); c = None if C is None else torch.tensor(np.nan_to_num(C[i:i + 8192]), device=DEVICE); acc = 0
            for m in ms: acc = acc + (m(x, c) - x) ** 2 if fam != 'sb' else acc + (m(x) - x) ** 2
            out[i:i + 8192] = (acc / len(ms)).cpu().numpy()
        full = out
    full[:FIRST_VALID[fam]] = np.nan; return full

E1CACHE = {}; NORM = {}
def norm_for(sym, fam):
    if (sym, fam) not in NORM:
        E = E1CACHE.setdefault((sym, fam), errs(sym, fam, Z1[sym])); a, b = FIRST_VALID[fam], BOUNDS['train'][1]
        NORM[(sym, fam)] = np.maximum(np.percentile(E[a:b].reshape(-1, 6), 99, axis=0), 1e-8)
    return NORM[(sym, fam)]
TEMPORAL = ['mean', 'max', 'q90', 'cur']; FEATAGG = ['fmean', 'fmax']
def fam_scorers(fam): return [f'w15|{t}|{f}' for t in TEMPORAL for f in FEATAGG] if fam == 'w15' else [f'{fam}|cur|{f}' for f in FEATAGG]
def score_family(sym, fam, Z, E=None):
    E = errs(sym, fam, Z) if E is None else E; nm = norm_for(sym, fam); out = {}
    if fam == 'w15':
        for f in FEATAGG:
            P = E.mean(axis=2) if f == 'fmean' else (E / nm).max(axis=2)
            out[f'w15|mean|{f}'] = P.mean(axis=1); out[f'w15|max|{f}'] = P.max(axis=1); out[f'w15|q90|{f}'] = np.quantile(P, 0.9, axis=1); out[f'w15|cur|{f}'] = P[:, -1]
    else:
        out[f'{fam}|cur|fmean'] = E.mean(axis=1); out[f'{fam}|cur|fmax'] = (E / nm).max(axis=1)
    return {k: v.astype(np.float32) for k, v in out.items()}
def rz_score(Z): z = np.abs(Z).max(axis=1).astype(np.float32); z[:WARM] = np.nan; return z
SIMP = {'sb': 0, 'ctx': 1, 'dn': 1, 'w15': 2}
def fam_of(scorer): return scorer.split('|')[0]

SC1 = {}                                   # (sym, scorer) -> Layer 1 score array (genuine data, all bars)
for sym in SYMS:
    for fam in ('w15', 'sb'):
        for k, v in score_family(sym, fam, Z1[sym], E1CACHE.setdefault((sym, fam), errs(sym, fam, Z1[sym]))).items(): SC1[(sym, k)] = v
    SC1[(sym, 'rz')] = rz_score(Z1[sym])
# v1 is reproduced exactly: the v1 rule (mean over all 90 errors) and the v1 single-bar rule, plus the robust-z baseline
ok_rep = True
for sym, tag in SYMS.items():
    for scorer, fn in (('w15|mean|fmean', 'window15'), ('sb|cur|fmean', 'single_bar'), ('rz', 'baseline_robust_z')):
        ref = pd.read_parquet(V1 / 'scores' / f'{tag}_{fn}_scores_l1.parquet')['score'].to_numpy(); new = SC1[(sym, scorer)]; m = np.isfinite(ref) & np.isfinite(new)
        ok_rep &= bool(m.sum() > 100000 and np.allclose(new[m], ref[m], rtol=2e-3, atol=1e-5) and np.array_equal(np.isfinite(ref), np.isfinite(new)))
assert ok_rep, 'v1 scores not reproduced'; print('v1 Layer 1 scores reproduced from the saved v1 models (window15 mean, single_bar, robust-z).'); TS['v1_reproduced'] = UTC()

v1 Layer 1 scores reproduced from the saved v1 models (window15 mean, single_bar, robust-z).


## 4 · Matched alarm burden: thresholds from Layer 1 VALIDATION so that genuine alarm *episodes/day* equal the budget

In [5]:
EP_GAP = 5; VAL = slice(*BOUNDS['val']); BUDGETS = PROTOCOL['budgets_episodes_per_day']; PB = PROTOCOL['primary_budget']
def episodes(alarm, gap=EP_GAP):
    idx = np.flatnonzero(alarm); return 0 if len(idx) == 0 else int(1 + (np.diff(idx) > gap).sum())
PGRID = 1 - np.logspace(-4.6, -0.3, 260)
def thr_for_budget(v, budget):
    v = v[np.isfinite(v)]; days = len(v) / 1440; cand = np.unique(np.quantile(v, PGRID)); ok = [t for t in cand if episodes(v > t) / days <= budget]
    return float(min(ok)) if ok else float(v.max())
def budget_thresholds(scores): return {b: thr_for_budget(scores[VAL], b) for b in BUDGETS}
THRB = {}
def register_thresholds(keys):
    for k in keys: THRB[k] = budget_thresholds(SC1[k])
register_thresholds(list(SC1)); TAIL_A = WARM + int(0.9 * (BOUNDS['train'][1] - WARM)) + 15
print({f'{k[0][:3]}|{k[1]}': {b: round(t, 3) for b, t in v.items()} for k, v in THRB.items() if k[1] in ('w15|mean|fmean', 'sb|cur|fmean', 'w15|max|fmax')})

{'BTC|w15|mean|fmean': {1: 9.365, 2: 5.5, 4: 3.504}, 'BTC|w15|max|fmax': {1: 16.975, 2: 12.253, 4: 7.902}, 'BTC|sb|cur|fmean': {1: 1.277, 2: 1.102, 4: 0.878}, 'ETH|w15|mean|fmean': {1: 7.328, 2: 5.746, 4: 3.538}, 'ETH|w15|max|fmax': {1: 14.491, 2: 10.908, 4: 7.462}, 'ETH|sb|cur|fmean': {1: 2.332, 2: 1.629, 4: 1.343}}


## 5 · Development probe on Layer 1 VALIDATION (no Layer 3 label touched)
117 short pseudo-anomalies per symbol: shifts of 1-3 random features by ±magnitude × half-sine, in 9 strata (3 durations × 3 magnitudes, 13 events each), placed ≥ 60 bars apart. It ranks *sensitivity to short and sustained feature deviations*; it is not evidence of manipulation detection.

In [6]:
PRB = PROTOCOL['development_probe']; DURB = [(1, 3), (4, 12), (13, 40)]; MAGS = [2, 4, 8]; STRATA = [(di, mi) for di in range(3) for mi in range(3)]
def build_probe(sym_i, sym):
    rng = np.random.default_rng(PRB['seed'] + sym_i); evs = []
    for si, (di, mi) in enumerate(STRATA):
        for _ in range(13):
            L = int(rng.integers(DURB[di][0], DURB[di][1] + 1)); k = int(rng.integers(1, 4)); evs.append({'stratum': si, 'dur_bin': di, 'mag': MAGS[mi], 'L': L, 'features': rng.choice(6, k, replace=False).tolist(), 'signs': rng.choice([-1, 1], k).tolist()})
    a, b = BOUNDS['val']; placed = []
    for i in sorted(range(len(evs)), key=lambda i: -evs[i]['L']):
        L = evs[i]['L']
        for _ in range(50000):
            s = int(rng.integers(a, b - 42))
            if all(s >= p[0] + p[1] + 60 or s + L + 60 <= p[0] for p in placed): break
        else: raise RuntimeError('probe placement failed')
        placed.append((s, L)); evs[i]['start'] = s
    Zp = Z1[sym].astype(np.float64).copy()
    for e in evs:
        env = np.sin(np.pi * (np.arange(e['L']) + 0.5) / e['L'])
        for f, sg in zip(e['features'], e['signs']): Zp[e['start']:e['start'] + e['L'], f] += sg * e['mag'] * env
    neg = np.zeros(N_BARS, bool); neg[a:b] = True
    for e in evs: neg[e['start']:e['start'] + e['L'] + 15] = False
    return evs, np.clip(Zp, -20, 20).astype(np.float32), neg
PROBE = {}
for i, sym in enumerate(SYMS):
    evs, Zp, neg = build_probe(i, sym); PROBE[sym] = {'events': evs, 'Z': Zp, 'neg': neg}
    st = sorted((e['start'], e['L']) for e in evs); assert len(evs) == 117 and all(st[j + 1][0] >= st[j][0] + st[j][1] + 60 for j in range(len(st) - 1))
print('probe built:', {s: (len(v['events']), int(v['neg'].sum())) for s, v in PROBE.items()}); TS['probe_built'] = UTC()
SELECTION = {'done': False}
def probe_rows(sym, scorer, sp):
    out = []; ev = PROBE[sym]['events']
    for b, t in THRB[(sym, scorer)].items():
        det = np.array([bool((sp[e['start']:e['start'] + e['L']] > t).any()) for e in ev])
        for si in range(len(STRATA)):
            m = np.array([e['stratum'] == si for e in ev]); out.append({'sym': sym, 'scorer': scorer, 'budget': int(b), 'stratum': si, 'dur_bin': STRATA[si][0], 'mag': MAGS[STRATA[si][1]], 'detected': int(det[m].sum()), 'n': int(m.sum())})
    return out
PROBE_ROWS = []
def probe_family(fam):
    for sym in SYMS:
        for k, sp in score_family(sym, fam, PROBE[sym]['Z']).items(): PROBE_ROWS.extend(probe_rows(sym, k, sp))
for fam in ('w15', 'sb'): probe_family(fam)
TS['probe_scored_v1'] = UTC()
def probe_table():
    d = pd.DataFrame(PROBE_ROWS); d['recall'] = d.detected / d.n; g = d[d.budget == PB].groupby(['scorer', 'sym']).recall.mean().unstack(); g['probe'] = g.mean(axis=1)
    g['short(1-3)'] = d[(d.budget == PB) & (d.dur_bin == 0)].groupby('scorer').recall.mean(); g['mid(4-12)'] = d[(d.budget == PB) & (d.dur_bin == 1)].groupby('scorer').recall.mean(); g['long(13-40)'] = d[(d.budget == PB) & (d.dur_bin == 2)].groupby('scorer').recall.mean()
    return g.sort_values('probe', ascending=False)
PT = probe_table(); display(PT)

probe built: {'BTCUSDT': (117, 15619), 'ETHUSDT': (117, 15647)}


sym,BTCUSDT,ETHUSDT,probe,short(1-3),mid(4-12),long(13-40)
scorer,,,,,,
sb|cur|fmax,0.7607,0.6325,0.6966,0.6026,0.7051,0.7821
sb|cur|fmean,0.7094,0.6154,0.6624,0.5641,0.7051,0.7179
w15|mean|fmax,0.3333,0.3761,0.3547,0.1026,0.3718,0.5897
w15|cur|fmax,0.3333,0.359,0.3462,0.2564,0.4103,0.3718
w15|q90|fmax,0.3162,0.3333,0.3248,0.1026,0.3846,0.4872
w15|max|fmax,0.2137,0.265,0.2393,0.1795,0.2821,0.2564
w15|mean|fmean,0.188,0.2051,0.1966,0,0.2179,0.3718
w15|q90|fmean,0.1624,0.1709,0.1667,0.01282,0.2564,0.2308
w15|max|fmean,0.1111,0.09402,0.1026,0.0641,0.07692,0.1667


## 6 · Sufficiency gate: is better *scoring* of the existing 15-bar model enough?
Pre-registered: the architecture stage runs **only if** the best 15-bar scoring rule does not reach the v1 single-bar rule on the Layer 1 probe at the primary budget.

In [7]:
BEST_W15 = PT[PT.index.str.startswith('w15')].probe.max(); SB_BASE = float(PT.loc['sb|cur|fmean', 'probe']); ARCH_RUN = bool(BEST_W15 < SB_BASE)
GATE_INFO = {'best_w15_scorer': PT[PT.index.str.startswith('w15')].probe.idxmax(), 'best_w15_probe': float(BEST_W15), 'v1_single_bar_probe': SB_BASE, 'architecture_stage_runs': ARCH_RUN}
print(GATE_INFO); TS['gate_decided'] = UTC()

{'best_w15_scorer': 'w15|mean|fmax', 'best_w15_probe': 0.3547008547008547, 'v1_single_bar_probe': 0.6623931623931625, 'architecture_stage_runs': True}


## 7 · Architecture stage (only if the gate says scoring is insufficient): `ctx_bar` and `dn_bar`, Layer 1 TRAIN only

In [8]:
TRAIN_END = BOUNDS['train'][1]; MAX_EP, PATIENCE, BS = 150, 10, 512; TRAIN_LOG2 = []
def train_cae(fam, sym, seed):
    d_ctx = 12 if fam == 'ctx' else 0; Z = Z1[sym]; fv = FIRST_VALID[fam]; C = ctx_feats(Z) if fam == 'ctx' else None
    ends = np.arange(fv, TRAIN_END); cut_end = WARM + int(0.9 * (TRAIN_END - WARM)); fit = ends <= cut_end; es = ends >= cut_end + 15
    assert ends[fit].max() < TRAIN_END and ends[es].max() < TRAIN_END
    tx = lambda sel: torch.tensor(Z[ends[sel]], device=DEVICE); tc = (lambda sel: torch.tensor(C[ends[sel]], device=DEVICE)) if C is not None else (lambda sel: None)
    Xf, Xe, Cf, Ce = tx(fit), tx(es), tc(fit), tc(es)
    torch.manual_seed(seed); np.random.seed(seed); g = torch.Generator().manual_seed(seed); m = CAE(d_ctx).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, hist = float('inf'), 0, []
    for ep in range(MAX_EP):
        m.train(); perm = torch.randperm(len(Xf), generator=g).to(DEVICE); tl = 0.0
        for i in range(0, len(perm), BS):
            ix = perm[i:i + BS]; b = Xf[ix]; c = None if Cf is None else Cf[ix]; xin = b + 0.15 * torch.randn_like(b) if fam == 'dn' else b
            loss = ((m(xin, c) - b) ** 2).mean(); opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 5.0); opt.step(); tl += loss.item() * len(b)
        m.eval()
        with torch.no_grad(): ve = float(np.mean([float(((m(Xe[i:i + 8192], None if Ce is None else Ce[i:i + 8192]) - Xe[i:i + 8192]) ** 2).mean()) for i in range(0, len(Xe), 8192)]))
        hist.append((ep, tl / len(Xf), ve))
        if ve < best - 1e-6: best, bad, best_state, best_ep = ve, 0, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}, ep
        else:
            bad += 1
            if bad >= PATIENCE: break
    m.load_state_dict(best_state); m.eval()
    return m, {'sym': sym, 'family': fam, 'seed': seed, 'fit_windows': int(fit.sum()), 'es_windows': int(es.sum()), 'epochs_run': len(hist), 'best_epoch': best_ep, 'best_es_loss': best, 'first_end_fit': int(ends[fit].min()), 'last_end_fit': int(ends[fit].max()), 'last_end_es': int(ends[es].max())}
if ARCH_RUN:
    TS['arch_train_start'] = UTC(); t0 = time.time()
    for fam in ('ctx', 'dn'):
        for sym in SYMS:
            ms = []
            for sd in SEEDS:
                m, rep = train_cae(fam, sym, sd); ms.append(m); TRAIN_LOG2.append(rep); print(f"{sym} {fam} seed {sd}: epochs {rep['epochs_run']:3d} best {rep['best_epoch']:3d} es-loss {rep['best_es_loss']:.4f} ({time.time()-t0:.0f}s)")
            NEWM[(sym, fam)] = ms
            for k, v in score_family(sym, fam, Z1[sym], E1CACHE.setdefault((sym, fam), errs(sym, fam, Z1[sym]))).items(): SC1[(sym, k)] = v
    register_thresholds([k for k in SC1 if fam_of(k[1]) in ('ctx', 'dn')])
    for fam in ('ctx', 'dn'): probe_family(fam)
    TS['arch_train_end'] = UTC(); PT = probe_table()
else: print('Architecture stage skipped: 15-bar scoring is sufficient on the probe.')
display(PT)

BTCUSDT ctx seed 11: epochs 118 best 107 es-loss 0.0423 (62s)
BTCUSDT ctx seed 22: epochs 150 best 139 es-loss 0.0411 (134s)
BTCUSDT ctx seed 33: epochs 102 best  91 es-loss 0.0440 (181s)
ETHUSDT ctx seed 11: epochs  45 best  34 es-loss 0.0539 (206s)
ETHUSDT ctx seed 22: epochs  93 best  82 es-loss 0.0513 (250s)
ETHUSDT ctx seed 33: epochs  69 best  58 es-loss 0.0519 (283s)
BTCUSDT dn seed 11: epochs  95 best  84 es-loss 0.0789 (328s)
BTCUSDT dn seed 22: epochs  97 best  86 es-loss 0.0783 (374s)
BTCUSDT dn seed 33: epochs  78 best  67 es-loss 0.0800 (411s)
ETHUSDT dn seed 11: epochs  71 best  60 es-loss 0.1005 (444s)
ETHUSDT dn seed 22: epochs  48 best  37 es-loss 0.1100 (467s)
ETHUSDT dn seed 33: epochs  87 best  76 es-loss 0.0961 (507s)


sym,BTCUSDT,ETHUSDT,probe,short(1-3),mid(4-12),long(13-40)
scorer,,,,,,
ctx|cur|fmax,0.8462,0.7436,0.7949,0.6923,0.8333,0.859
ctx|cur|fmean,0.8291,0.7009,0.765,0.6923,0.7821,0.8205
dn|cur|fmax,0.7778,0.641,0.7094,0.6026,0.7436,0.7821
sb|cur|fmax,0.7607,0.6325,0.6966,0.6026,0.7051,0.7821
dn|cur|fmean,0.7179,0.6239,0.6709,0.5769,0.7308,0.7051
sb|cur|fmean,0.7094,0.6154,0.6624,0.5641,0.7051,0.7179
w15|mean|fmax,0.3333,0.3761,0.3547,0.1026,0.3718,0.5897
w15|cur|fmax,0.3333,0.359,0.3462,0.2564,0.4103,0.3718
w15|q90|fmax,0.3162,0.3333,0.3248,0.1026,0.3846,0.4872


## 8 · Model selection from the Layer 1 probe + burden-transfer gate (no Layer 3 label)

In [9]:
def tail_burden(sym, scorer):
    t = THRB[(sym, scorer)][PB]; v = SC1[(sym, scorer)][TAIL_A:TRAIN_END]; return episodes(v > t) / (len(v) / 1440)
def select_rule(tab):
    t = tab[tab.gate_ok]; assert len(t), 'no candidate passes the burden gate'; best = t.probe.max(); tied = t[t.probe >= best - 0.02]; tied = tied[tied.simp_rank == tied.simp_rank.min()]
    return tied.sort_values(['probe', 'scorer'], ascending=[False, True]).iloc[0].scorer
SELTAB = pd.DataFrame({'scorer': PT.index, 'probe': PT.probe.values, 'simp_rank': [SIMP[fam_of(s)] for s in PT.index],
                       'tail_ep_day_btc': [tail_burden('BTCUSDT', s) for s in PT.index], 'tail_ep_day_eth': [tail_burden('ETHUSDT', s) for s in PT.index]})
SELTAB['gate_ok'] = (SELTAB[['tail_ep_day_btc', 'tail_ep_day_eth']].max(axis=1) <= 3 * PB); SELECTED = select_rule(SELTAB); SELECTION.update({'done': True, 'selected': SELECTED, 'utc': UTC()}); TS['selection_done'] = UTC()
display(SELTAB.set_index('scorer')); print('SELECTED by the probe rule (no Layer 3 label used):', SELECTED)

,probe,simp_rank,tail_ep_day_btc,tail_ep_day_eth,gate_ok
scorer,,,,,
ctx|cur|fmax,0.7949,1,0.636,1.431,True
ctx|cur|fmean,0.765,1,1.272,1.908,True
dn|cur|fmax,0.7094,1,0.318,0.477,True
sb|cur|fmax,0.6966,0,0,0.477,True
dn|cur|fmean,0.6709,1,0.7951,0.636,True
sb|cur|fmean,0.6624,0,0.477,0.7951,True
w15|mean|fmax,0.3547,2,0.318,0.159,True
w15|cur|fmax,0.3462,2,0.159,0.159,True
w15|q90|fmax,0.3248,2,0.159,0.159,True


SELECTED by the probe rule (no Layer 3 label used): ctx|cur|fmax


## 9 · Layer 3 VALIDATION: dilution diagnosis and model comparison (descriptive; test still locked)

In [10]:
SC3, E3 = {}, {}
for sym in SYMS:
    fams = ['w15', 'sb'] + (['ctx', 'dn'] if ARCH_RUN else [])
    for fam in fams:
        E3[(sym, fam)] = errs(sym, fam, Z3[sym]) if fam == 'w15' else None
        for k, v in score_family(sym, fam, Z3[sym], E3[(sym, fam)]).items(): SC3[(sym, k)] = v
    SC3[(sym, 'rz')] = rz_score(Z3[sym])
V1MAP = {'w15|mean|fmean': 'window15', 'sb|cur|fmean': 'single_bar', 'rz': 'baseline_robust_z'}
def ops_for(sym, scorer):
    o = {f'b{b}': t for b, t in THRB[(sym, scorer)].items()}
    if scorer in V1MAP: o['v1_q99.5'] = float(V1THR[f'{sym}|{V1MAP[scorer]}']['q99.5'])
    return o
TEST_STATE = {'unlocked': False, 'unlock_utc': None, 'calls': 0}
def l3_eval(sym, split, scorer, ci=False):
    assert SELECTION['done'], 'Layer 3 is locked until selection is made'
    if split == 'test': assert TEST_STATE['unlocked'], 'test split is locked'; TEST_STATE['calls'] += 1
    scores = SC3[(sym, scorer)]; ops = ops_for(sym, scorer); a, b = BOUNDS[split]; lb = LB3[sym]; idx = np.arange(N_BARS); ev = EV3[(EV3.symbol == sym) & (EV3.split == split)].reset_index(drop=True)
    ok = (idx >= a) & (idx < b) & np.isfinite(scores); pos = ok & (lb.synthetic_label.to_numpy() == 1); neg = ok & (lb.normal_background.to_numpy() == 1); m = pos | neg; y = pos[m].astype(int); s = scores[m]
    res = {'n_events': int(len(ev)), 'n_pos_bars': int(pos.sum()), 'n_neg_bars': int(neg.sum()), 'roc_auc': auc_rank(s, y), 'pr_auc': avg_precision(s, y), 'pr_auc_chance': float(y.mean())}
    if ci:
        rng = np.random.default_rng(PROTOCOL['bootstrap']['seed']); evb = [np.arange(r.start_idx, r.end_idx + 1) for r in ev.itertuples()]; negs = scores[neg]; ba, bp = [], []
        for _ in range(PROTOCOL['bootstrap']['B']):
            pick = rng.integers(0, len(ev), len(ev)); ps = np.concatenate([scores[evb[i]] for i in pick]); ps = ps[np.isfinite(ps)]; yy = np.r_[np.ones(len(ps), int), np.zeros(len(negs), int)]; xx = np.r_[ps, negs]; ba.append(auc_rank(xx, yy)); bp.append(avg_precision(xx, yy))
        res['roc_auc_ci95'] = [float(np.nanpercentile(ba, 2.5)), float(np.nanpercentile(ba, 97.5))]; res['pr_auc_ci95'] = [float(np.nanpercentile(bp, 2.5)), float(np.nanpercentile(bp, 97.5))]
    dur = (ev.end_idx - ev.start_idx + 1).to_numpy(); cuts = np.quantile(dur, [1 / 3, 2 / 3]); dbin = np.digitize(dur, cuts, right=True); res['duration_tertile_edges_bars'] = [float(c) for c in cuts]; res['ops'] = {}
    for name, t in ops.items():
        al = scores > t; det = np.array([bool(al[r.start_idx:r.end_idx + 1].any()) for r in ev.itertuples()]); delay = np.array([int(np.argmax(al[r.start_idx:r.end_idx + 1])) if d_ else -1 for r, d_ in zip(ev.itertuples(), det)])
        negal = np.where(neg, al, False); days = max(neg.sum(), 1) / 1440; e2 = ev.assign(detected=det, dbin=dbin)
        fam = {k: [int(g.detected.sum()), len(g)] for k, g in e2.groupby('scenario_family')}
        res['ops'][name] = {'threshold': float(t), 'event_recall': float(det.mean()), 'event_recall_ci95': wilson(int(det.sum()), len(det)), 'events_detected': int(det.sum()), 'median_detection_delay_bars': float(np.median(delay[det])) if det.any() else None,
            'clean_negative_episodes_per_day': float(episodes(negal) / days), 'bar_recall': float(al[pos].mean()) if pos.any() else None, 'by_family': fam, 'families_covered': int(sum(1 for v in fam.values() if v[0] > 0)), 'families_total': len(fam),
            'by_severity': {k: [int(g.detected.sum()), len(g)] for k, g in e2.groupby('severity')}, 'by_ood': {str(k): [int(g.detected.sum()), len(g)] for k, g in e2.groupby('is_ood')}, 'by_duration_tertile': [[int(g.detected.sum()), len(g)] for _, g in e2.groupby('dbin')]}
    res['layer1'] = {}
    for name, t in ops.items():
        v = SC1[(sym, scorer)][a:b]; al = v > t; res['layer1'][name] = {'alarm_bars_per_day': float(al.sum() / ((b - a) / 1440)), 'episodes_per_day': float(episodes(al) / ((b - a) / 1440))}
    return res
CANDS = list(PT.index); DIAG = {f'{sym}|{sc}': l3_eval(sym, 'val', sc) for sym in SYMS for sc in CANDS + ['rz']}
DT_ = pd.DataFrame([{'scorer': sc, 'sym': sym[:3], 'recall@b2': DIAG[f'{sym}|{sc}']['ops']['b2']['event_recall'], 'PR-AUC': DIAG[f'{sym}|{sc}']['pr_auc'], 'ROC-AUC': DIAG[f'{sym}|{sc}']['roc_auc'],
    'short': DIAG[f'{sym}|{sc}']['ops']['b2']['by_duration_tertile'][0][0] / max(DIAG[f'{sym}|{sc}']['ops']['b2']['by_duration_tertile'][0][1], 1), 'long': DIAG[f'{sym}|{sc}']['ops']['b2']['by_duration_tertile'][-1][0] / max(DIAG[f'{sym}|{sc}']['ops']['b2']['by_duration_tertile'][-1][1], 1)} for sym in SYMS for sc in CANDS + ['rz']])
DIAG_TAB = DT_.pivot(index='scorer', columns='sym'); display(DIAG_TAB)
def top_share(E, idx): P = E[idx].mean(axis=2); return P.max(axis=1) / np.maximum(P.sum(axis=1), 1e-12)
DILUTION = {}
for sym in SYMS:
    a, b = BOUNDS['val']; idx = np.arange(N_BARS); fin = np.isfinite(SC1[(sym, 'w15|mean|fmean')]); gen = np.flatnonzero((idx >= a) & (idx < b) & fin); alarm = gen[SC1[(sym, 'w15|mean|fmean')][gen] > THRB[(sym, 'w15|mean|fmean')][PB]]
    ev_bars = np.flatnonzero((idx >= a) & (idx < b) & (LB3[sym].synthetic_label.to_numpy() == 1) & fin)
    DILUTION[sym] = {'uniform_share_if_no_concentration': 1 / 15, 'genuine_val_bars_median_top_position_share': float(np.median(top_share(E1CACHE[(sym, 'w15')], gen))), 'genuine_alarm_bars_median_top_position_share': float(np.median(top_share(E1CACHE[(sym, 'w15')], alarm))) if len(alarm) else None,
                     'layer3_event_bars_median_top_position_share': float(np.median(top_share(E3[(sym, 'w15')], ev_bars))), 'threshold_over_median_genuine_score': {sc: float(THRB[(sym, sc)][PB] / np.nanmedian(SC1[(sym, sc)][a:b])) for sc in ('w15|mean|fmean', 'w15|max|fmean', 'w15|cur|fmean', 'sb|cur|fmean')}}
print(json.dumps(DILUTION, indent=1, default=jsonable)); TS['diagnosis_done'] = UTC()

recall@b2          PR-AUC         ROC-AUC        short         long       
sym                  BTC     ETH     BTC     ETH     BTC    ETH   BTC     ETH  BTC    ETH
scorer                                                                                   
ctx|cur|fmax      0.5333  0.3667  0.1787  0.1531  0.7176  0.727   0.6  0.3636  0.4 0.5556
ctx|cur|fmean        0.5  0.4667  0.1698  0.1678   0.712 0.7341   0.6  0.4545  0.3 0.6667
dn|cur|fmax       0.1333  0.4333  0.1018  0.1189  0.6523  0.672   0.1  0.4545  0.2 0.7778
dn|cur|fmean      0.1667  0.3667  0.1083  0.1072   0.662 0.6691   0.3  0.3636  0.1 0.6667
rz                     0       0 0.08379 0.07696  0.6135 0.6279     0       0    0      0
sb|cur|fmax          0.1     0.4  0.1045   0.111  0.6543  0.664   0.1  0.3636  0.1 0.7778
sb|cur|fmean      0.1667     0.4  0.1141  0.1061  0.6643 0.6677   0.3  0.3636  0.1 0.7778
w15|cur|fmax         0.1  0.1667  0.1055  0.1007  0.6373 0.6564   0.1 0.09091    0 0.4444
w15|cur|fmean     0.1333 0.06667 0.09588 0.08912  0.6335   0.66   0.1       0  0.1 0.2222
w15|max|fmax      0.1333  0.1667 0.09881  0.1218  0.6636 0.7231   0.1  0.1818  0.1 0.2222
w15|max|fmean     0.1333  0.1667 0.08471 0.08709  0.6266 0.6749   0.1  0.1818  0.1 0.2222
w15|mean|fmax     0.1667  0.2333  0.1336  0.1716  0.6934 0.7498   0.1  0.1818  0.2 0.4444
w15|mean|fmean    0.1333  0.1667    0.11  0.1156  0.6485 0.7123   0.1  0.1818  0.1 0.2222
w15|q90|fmax      0.1333     0.2  0.1189  0.1521  0.6789 0.7509   0.1  0.1818  0.1 0.3333
w15|q90|fmean     0.1333  0.1667 0.09948  0.1089  0.6225 0.7042   0.1  0.1818  0.1 0.2222

{
 "BTCUSDT": {
  "uniform_share_if_no_concentration": 0.06666666666666667,
  "genuine_val_bars_median_top_position_share": 0.1817583441734314,
  "genuine_alarm_bars_median_top_position_share": 0.272597074508667,
  "layer3_event_bars_median_top_position_share": 0.18037761747837067,
  "threshold_over_median_genuine_score": {
   "w15|mean|fmean": 6.062508583068848,
   "w15|max|fmean": 10.835145950317383,
   "w15|cur|fmean": 31.856473922729492,
   "sb|cur|fmean": 16.800479888916016
  }
 },
 "ETHUSDT": {
  "uniform_share_if_no_concentration": 0.06666666666666667,
  "genuine_val_bars_median_top_position_share": 0.17442432045936584,
  "genuine_alarm_bars_median_top_position_share": 0.24267202615737915,
  "layer3_event_bars_median_top_position_share": 0.1730218529701233,
  "threshold_over_median_genuine_score": {
   "w15|mean|fmean": 5.631402492523193,
   "w15|max|fmean": 10.048863410949707,
   "w15|cur|fmean": 26.232421875,
   "sb|cur|fmean": 15.436553001403809
  }
 }
}


## 10 · Recommendation (pre-registered fallback), VALIDATION results with CIs saved, then the single exploratory TEST pass

In [11]:
R_SEL = float(np.mean([DIAG[f'{sym}|{SELECTED}']['ops']['b2']['event_recall'] for sym in SYMS])); R_V1SB = float(np.mean([DIAG[f'{sym}|sb|cur|fmean']['ops']['b2']['event_recall'] for sym in SYMS]))
FALLBACK = bool(R_SEL < R_V1SB); RECOMMENDED = 'sb|cur|fmean' if FALLBACK else SELECTED
DECL = list(dict.fromkeys([SELECTED, 'w15|mean|fmean', 'sb|cur|fmean', 'rz']))
REC_INFO = {'selected_by_probe': SELECTED, 'layer3_val_recall_b2_selected': R_SEL, 'layer3_val_recall_b2_v1_single_bar': R_V1SB, 'fallback_triggered': FALLBACK, 'recommended': RECOMMENDED, 'declared_final_set': DECL}
print(REC_INFO)
TS['val_eval_start'] = UTC(); RES_VAL = {f'{sym}|{sc}': l3_eval(sym, 'val', sc, ci=True) for sym in SYMS for sc in DECL}
dump_json({'protocol_sha256': PROTOCOL_HASH, 'recommendation': REC_INFO, 'thresholds': {f'{k[0]}|{k[1]}': v for k, v in THRB.items()}, 'val_results': RES_VAL, 'saved_utc': UTC()}, STAGE / 'reports' / 'validation_results.json'); TS['val_saved'] = UTC()
assert (STAGE / 'reports' / 'validation_results.json').is_file() and not TEST_STATE['unlocked']
TEST_STATE['unlocked'] = True; TEST_STATE['unlock_utc'] = UTC(); TS['test_unlocked'] = TEST_STATE['unlock_utc']
RES_TEST = {f'{sym}|{sc}': l3_eval(sym, 'test', sc, ci=True) for sym in SYMS for sc in DECL}; TEST_STATE['unlocked'] = False; TS['test_evaluated'] = UTC()
print('TEST evaluated once for the declared set only (exploratory: v1 had already examined this split).')
def comp_table(R, op):
    rows = {}
    for k, v in R.items():
        o = v['ops'][op] if op in v['ops'] else None
        if o is None: continue
        rows[k] = {'ROC-AUC': round(v['roc_auc'], 3), 'PR-AUC': round(v['pr_auc'], 3), 'PR chance': round(v['pr_auc_chance'], 3), 'event recall': f"{o['events_detected']}/{v['n_events']} ({o['event_recall_ci95'][0]:.2f}-{o['event_recall_ci95'][1]:.2f})",
                   'delay(bars)': o['median_detection_delay_bars'], 'L1 ep/day': round(v['layer1'][op]['episodes_per_day'], 2), 'L3 clean ep/day': round(o['clean_negative_episodes_per_day'], 2), 'families': f"{o['families_covered']}/{o['families_total']}"}
    return pd.DataFrame(rows).T
for nm, R in (('VALIDATION', RES_VAL), ('TEST (exploratory)', RES_TEST)):
    for op in ('b2', 'v1_q99.5'): print(nm, op); display(comp_table(R, op))

{'selected_by_probe': 'ctx|cur|fmax', 'layer3_val_recall_b2_selected': 0.44999999999999996, 'layer3_val_recall_b2_v1_single_bar': 0.2833333333333333, 'fallback_triggered': False, 'recommended': 'ctx|cur|fmax', 'declared_final_set': ['ctx|cur|fmax', 'w15|mean|fmean', 'sb|cur|fmean', 'rz']}
TEST evaluated once for the declared set only (exploratory: v1 had already examined this split).
VALIDATION b2


,ROC-AUC,PR-AUC,PR chance,event recall,delay(bars),L1 ep/day,L3 clean ep/day,families
BTCUSDT|ctx|cur|fmax,0.718,0.179,0.049,16/30 (0.36-0.70),2,2,1.99,7/7
BTCUSDT|w15|mean|fmean,0.649,0.11,0.049,4/30 (0.05-0.30),7,2,1.99,3/7
BTCUSDT|sb|cur|fmean,0.664,0.114,0.049,5/30 (0.07-0.34),4,2,1.99,4/7
BTCUSDT|rz,0.614,0.084,0.049,0/30 (0.00-0.11),None,0,0,0/7
ETHUSDT|ctx|cur|fmax,0.727,0.153,0.046,11/30 (0.22-0.54),6,2,2.07,5/7
ETHUSDT|w15|mean|fmean,0.712,0.116,0.046,5/30 (0.07-0.34),11,1.92,2.07,4/7
ETHUSDT|sb|cur|fmean,0.668,0.106,0.046,12/30 (0.25-0.58),7,2,2.07,6/7
ETHUSDT|rz,0.628,0.077,0.046,0/30 (0.00-0.11),None,0,0,0/7


VALIDATION v1_q99.5


,ROC-AUC,PR-AUC,PR chance,event recall,delay(bars),L1 ep/day,L3 clean ep/day,families
BTCUSDT|w15|mean|fmean,0.649,0.11,0.049,3/30 (0.03-0.26),10,1.15,1.24,2/7
BTCUSDT|sb|cur|fmean,0.664,0.114,0.049,13/30 (0.27-0.61),6,6.31,6.63,6/7
BTCUSDT|rz,0.614,0.084,0.049,7/30 (0.12-0.41),5,6.15,6.21,6/7
ETHUSDT|w15|mean|fmean,0.712,0.116,0.046,3/30 (0.03-0.26),19,0.77,0.83,3/7
ETHUSDT|sb|cur|fmean,0.668,0.106,0.046,17/30 (0.39-0.73),7,6.23,6.36,6/7
ETHUSDT|rz,0.628,0.077,0.046,5/30 (0.07-0.34),8,6.15,6.53,4/7


TEST (exploratory) b2


,ROC-AUC,PR-AUC,PR chance,event recall,delay(bars),L1 ep/day,L3 clean ep/day,families
BTCUSDT|ctx|cur|fmax,0.731,0.266,0.085,18/36 (0.34-0.66),10.5,1.64,1.28,7/7
BTCUSDT|w15|mean|fmean,0.687,0.19,0.085,4/36 (0.04-0.25),20.5,0.57,0.56,2/7
BTCUSDT|sb|cur|fmean,0.692,0.214,0.085,8/36 (0.12-0.38),21,0.43,0.4,5/7
BTCUSDT|rz,0.628,0.135,0.085,0/36 (0.00-0.10),None,0,0,0/7
ETHUSDT|ctx|cur|fmax,0.703,0.179,0.064,10/36 (0.16-0.44),20.5,0.79,0.86,6/7
ETHUSDT|w15|mean|fmean,0.667,0.122,0.064,2/36 (0.02-0.18),43,0.5,0.55,2/7
ETHUSDT|sb|cur|fmean,0.648,0.133,0.064,8/36 (0.12-0.38),18.5,1.14,1.1,6/7
ETHUSDT|rz,0.615,0.098,0.064,0/36 (0.00-0.10),None,0,0,0/7


TEST (exploratory) v1_q99.5


,ROC-AUC,PR-AUC,PR chance,event recall,delay(bars),L1 ep/day,L3 clean ep/day,families
BTCUSDT|w15|mean|fmean,0.687,0.19,0.085,3/36 (0.03-0.22),18,0.29,0.32,1/7
BTCUSDT|sb|cur|fmean,0.692,0.214,0.085,16/36 (0.30-0.60),15,3.14,2.81,6/7
BTCUSDT|rz,0.628,0.135,0.085,9/36 (0.14-0.41),10,4.71,4.57,6/7
ETHUSDT|w15|mean|fmean,0.667,0.122,0.064,1/36 (0.00-0.14),26,0.21,0.24,1/7
ETHUSDT|sb|cur|fmean,0.648,0.133,0.064,11/36 (0.18-0.47),16,3.64,3.53,7/7
ETHUSDT|rz,0.615,0.098,0.064,9/36 (0.14-0.41),10,4.07,4.08,6/7


## 11 · Persist models, scores, reports and manifest (staging directory only)

In [12]:
FAMDIR = {'ctx': 'ctx_bar', 'dn': 'dn_bar'}
for (sym, fam), ms in NEWM.items():
    for sd, m in zip(SEEDS, ms):
        p = STAGE / 'models' / SYMS[sym] / FAMDIR[fam]; p.mkdir(parents=True, exist_ok=True); torch.save({k: v.cpu() for k, v in m.state_dict().items()}, p / f'seed{sd}.pt'); dump_json({'family': fam, 'd_ctx': m.d_ctx, 'bottleneck': 3, 'input': 'current bar (6)' + (' + mean/std of previous 15 bars (12)' if m.d_ctx else '')}, p / 'arch.json')
for sym in SYMS: dump_json({'features': FEATURES, 'log1p_columns': ['rush_order_density', 'large_trade_count'], 'median': PREP[sym]['median'], 'scale': PREP[sym]['scale'], 'clip': [-20, 20], 'fit_rows': PREP[sym]['fit_rows'], 'source': PREP[sym]['source'] + ' (identical to models_autoencoder_v1)'}, STAGE / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json')
fname = lambda sc: sc.replace('|', '_')
for sym in SYMS:
    for sc in DECL:
        for src, store in (('l1', SC1), ('l3', SC3)): pd.DataFrame({'datetime': DT[sym], 'score': store[(sym, sc)].astype(np.float32)}).to_parquet(STAGE / 'scores' / f'{SYMS[sym]}_{fname(sc)}_scores_{src}.parquet', index=False)
FAMLOC = {'sb': ('models_autoencoder_v1', 'single_bar'), 'w15': ('models_autoencoder_v1', 'window15'), 'ctx': ('models_autoencoder_v2', 'ctx_bar'), 'dn': ('models_autoencoder_v2', 'dn_bar')}
def spec(sc):
    fam = fam_of(sc)
    if sc == 'rz': return {'scorer': 'rz', 'definition': 'max over features of |scaled value| (baseline, no model)'}
    t, f = sc.split('|')[1], sc.split('|')[2]
    return {'scorer': sc, 'family': fam, 'temporal_aggregator': t, 'feature_aggregator': f, 'model_dir': FAMLOC[fam][0] + '/models/{btc|eth}/' + FAMLOC[fam][1], 'seeds': SEEDS, 'ensemble': 'mean of squared per-position errors over the 3 seeds',
            'feature_error_scale_train_p99': {SYMS[s]: norm_for(s, fam).tolist() for s in SYMS} if f == 'fmax' else None, 'thresholds_episodes_per_day_budget': {SYMS[s]: {str(b): t_ for b, t_ in THRB[(s, sc)].items()} for s in SYMS}}
dump_json({'recommended': spec(RECOMMENDED), 'selected_by_probe': spec(SELECTED), 'recommendation_info': REC_INFO, 'declared_final_set': DECL, 'preprocessing': 'preprocessing/{btc|eth}_preprocessing.json', 'threshold_source': 'Layer 1 VALIDATION, largest alarm set with genuine episodes/day <= budget'}, STAGE / 'models' / 'selected_model.json')
dump_json({f'{k[0]}|{k[1]}': {str(b): t for b, t in v.items()} for k, v in THRB.items()}, STAGE / 'models' / 'thresholds_by_budget.json')
dump_json({'protocol_sha256': PROTOCOL_HASH, 'probe_rows': PROBE_ROWS, 'probe_events': {s: PROBE[s]['events'] for s in SYMS}, 'probe_table_budget2': PT.reset_index().to_dict(orient='records'), 'gate': GATE_INFO}, STAGE / 'reports' / 'probe_report.json')
dump_json({'selection_table': SELTAB.to_dict(orient='records'), 'selected': SELECTED, 'rule': PROTOCOL['selection_rule']}, STAGE / 'reports' / 'selection_table.json')
dump_json({'validation_all_candidates': DIAG, 'dilution': DILUTION, 'protocol_sha256': PROTOCOL_HASH, 'note': 'descriptive, Layer 3 VALIDATION only; not used for selection'}, STAGE / 'reports' / 'diagnosis_report.json')
dump_json({'training': TRAIN_LOG2, 'timestamps': TS, 'device': DEVICE, 'torch': torch.__version__, 'architecture_stage_ran': ARCH_RUN}, STAGE / 'reports' / 'training_report.json')
dump_json({'protocol_sha256': PROTOCOL_HASH, 'v1_protocol_sha256': V1_HASH, 'recommendation': REC_INFO, 'validation': RES_VAL, 'test_exploratory': RES_TEST, 'test_unlock_utc': TEST_STATE['unlock_utc'], 'test_evaluation_calls': TEST_STATE['calls']}, STAGE / 'reports' / 'evaluation_report.json')
LIM = ['Layer 3 labels are synthetic injection mechanisms; results do not prove detection of real manipulation. Spoofing, layering and liquidity withdrawal are not covered (order-book/TGNN track).',
       'The development probe is a set of generic feature-space perturbations without cross-feature coherence; it ranks sensitivity and can favour single-bar scorers. It is not evidence of manipulation detection.',
       'Layer 3 TEST had already been examined in v1; v2 test numbers are exploratory and not independent confirmation. Only the pre-declared final set was evaluated on test.',
       'Matched-burden thresholds come from only ~13 days of Layer 1 validation; episode budgets of 1-4 per day correspond to 13-52 episodes, so thresholds carry sampling noise. Layer 1 test drift can change test alarm rates.',
       'Layer 1 training data is normal-dominant, not verified normal-only; alarms on genuine data are not proven false alarms.',
       'Scores are uncalibrated anomaly signals; conformal calibration is a later stage. Three seeds only; per-seed variance of v2 candidates was not analysed.',
       'Event counts per split are modest (30 val / 36 test per symbol); intervals are wide and small differences are not reliable.']
(STAGE / 'reports' / 'limitations.md').write_text('# Limitations\n\n' + '\n'.join('- ' + l for l in LIM) + '\n', encoding='utf-8')
(STAGE / 'README.md').write_text('\n'.join(['# models_autoencoder_v2', '', 'One targeted improvement cycle on the Autoencoder. protocol/improvement_protocol.json was frozen (sha256 ' + PROTOCOL_HASH + ') before any v2 score, probe or model existed.', '',
    '- models/selected_model.json: recommended scorer, selected-by-probe scorer, thresholds per alarm-episode budget (1/2/4 per day), fallback decision', '- models/{btc,eth}/{ctx_bar,dn_bar}/ : only if the architecture stage ran', '- reports/{probe_report,selection_table,diagnosis_report,validation_results,evaluation_report,training_report}.json, limitations.md',
    '- scores/ : Layer 1 and Layer 3 scores of the declared final set', '', 'v1 models stay in models_autoencoder_v1 (unchanged); v2 scorers of v1 models are defined in selected_model.json. Read reports/limitations.md before quoting any number.']), encoding='utf-8')
def build_manifest():
    return {str(p.relative_to(STAGE)): {'size': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(STAGE.rglob('*')) if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
MANIFEST = {'artifact': 'models_autoencoder_v2', 'protocol_sha256': PROTOCOL_HASH, 'excluded': ['manifest.json', 'validation_report.json'], 'files': build_manifest()}
dump_json(MANIFEST, STAGE / 'manifest.json'); print('manifest entries:', len(MANIFEST['files']))

manifest entries: 45


## 12 · Validation of the NEW artifacts only, publish, and SHARE BLOCK

In [13]:
RES = []
def T(n, name, ok, detail=''):
    RES.append({'test': n, 'name': name, 'pass': bool(ok), 'detail': str(detail)[:300]}); print(('PASS ' if ok else 'FAIL ') + f'T{n:02d} {name}' + ('' if ok else f'  <-- {str(detail)[:200]}'))
req = ['protocol/improvement_protocol.json', 'models/selected_model.json', 'models/thresholds_by_budget.json', 'reports/probe_report.json', 'reports/selection_table.json', 'reports/diagnosis_report.json', 'reports/validation_results.json', 'reports/evaluation_report.json', 'reports/training_report.json',
       'reports/limitations.md', 'README.md', 'manifest.json'] + [f'preprocessing/{SYMS[s]}_preprocessing.json' for s in SYMS] + [f'scores/{SYMS[s]}_{fname(sc)}_scores_{src}.parquet' for s in SYMS for sc in DECL for src in ('l1', 'l3')] + \
      [f'models/{SYMS[s]}/{FAMDIR[f]}/seed{sd}.pt' for (s, f) in NEWM for sd in SEEDS]
T(1, 'all required files exist', all((STAGE / r).is_file() for r in req), [r for r in req if not (STAGE / r).is_file()][:5])
T(2, 'frozen sources unchanged (Layers 1, 2, 3 v1/v2, models_autoencoder_v1)', {n: pin_dir(BASE / n) for n in PIN} == PIN)
pj = json.loads((STAGE / 'protocol' / 'improvement_protocol.json').read_text()); body = {k: v for k, v in pj.items() if k != 'integrity'}
order = ['protocol_frozen', 'data_loaded', 'v1_reproduced', 'probe_built', 'probe_scored_v1', 'gate_decided'] + (['arch_train_start', 'arch_train_end'] if ARCH_RUN else []) + ['selection_done', 'diagnosis_done', 'val_saved', 'test_unlocked', 'test_evaluated']
T(3, 'protocol intact (hash matches) and frozen BEFORE any score/probe/model; Layer 3 diagnosis and test only after selection', hashlib.sha256(json.dumps(body, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode()).hexdigest() == PROTOCOL_HASH == pj['integrity']['sha256_of_canonical_json_excluding_this_field'] and all(TS[a] <= TS[b] for a, b in zip(order, order[1:])), {k: TS[k] for k in order if k in TS})
ok4 = True
for sym in SYMS:
    X = np.asarray(F1[sym], float).copy(); X[:, 3] = np.log1p(np.maximum(X[:, 3], 0)); X[:, 5] = np.log1p(np.maximum(X[:, 5], 0)); tr = X[15:90720]; med = np.median(tr, axis=0); iq = np.subtract(*np.percentile(tr, [75, 25], axis=0)) / 1.349
    sc_ = np.maximum(np.where(iq > 1e-12, iq, tr.std(axis=0)), 1e-8)
    for d_ in (STAGE, V1):
        pp = json.loads((d_ / 'preprocessing' / f'{SYMS[sym]}_preprocessing.json').read_text()); ok4 &= bool(np.allclose(pp['median'], med, rtol=1e-9) and np.allclose(pp['scale'], sc_, rtol=1e-9) and pp['fit_rows'] == [15, 90720])
T(4, 'preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute matches v2 and v1 files)', ok4)
ok5 = True
for sym, tag in SYMS.items():
    for scorer, fn in (('w15|mean|fmean', 'window15'), ('sb|cur|fmean', 'single_bar')):
        ref = pd.read_parquet(V1 / 'scores' / f'{tag}_{fn}_scores_l3.parquet')['score'].to_numpy(); new = SC3[(sym, scorer)]; m = np.isfinite(ref) & np.isfinite(new); ok5 &= bool(m.sum() > 100000 and np.allclose(new[m], ref[m], rtol=2e-3, atol=1e-5))
T(5, 'v1 models reload and reproduce the saved v1 Layer 1 and Layer 3 scores', ok5)
ok6, moved = True, False
for fam in ['w15', 'sb'] + (['ctx', 'dn'] if ARCH_RUN else []):
    sym = 'BTCUSDT'; Zs = Z1[sym][:6000].copy(); Zq = Zs.copy(); Zq[3000:] += 5.0; a = score_family(sym, fam, Zs); b = score_family(sym, fam, Zq)
    for k in a: ok6 &= bool(np.allclose(a[k][:3000], b[k][:3000], equal_nan=True, atol=1e-7)); moved |= bool(not np.allclose(a[k][3000:], b[k][3000:], equal_nan=True))
T(6, 'all scorers are causal (changing bars after t leaves scores at <= t unchanged) and react to later changes', ok6 and moved)
T(7, 'new-model training and early-stopping windows lie inside Layer 1 TRAIN and after the first legal bar', all(r['last_end_fit'] < TRAIN_END and r['last_end_es'] < TRAIN_END and r['first_end_fit'] >= FIRST_VALID[r['family']] and r['fit_windows'] > 1000 and r['es_windows'] > 100 for r in TRAIN_LOG2) if TRAIN_LOG2 else True, 'no new models trained' if not TRAIN_LOG2 else '')
thj = json.loads((STAGE / 'models' / 'thresholds_by_budget.json').read_text()); ok8 = True; days = (BOUNDS['val'][1] - BOUNDS['val'][0]) / 1440
for (sym, sc), tb in THRB.items():
    v = SC1[(sym, sc)][BOUNDS['val'][0]:BOUNDS['val'][1]]
    for b, t in tb.items():
        idx = np.flatnonzero(v > t); n_ep = 0 if len(idx) == 0 else 1 + sum(1 for i in range(1, len(idx)) if idx[i] - idx[i - 1] > 5); ok8 &= bool(n_ep / days <= b + 1e-9 and np.isclose(thj[f'{sym}|{sc}'][str(b)], t))
        lower = np.unique(np.quantile(v, PGRID)); lower = lower[lower < t]; ok8 &= bool(all(episodes(v > x) / days > b for x in lower))
T(8, 'every budget threshold recomputed from Layer 1 VALIDATION: episodes/day <= budget and no lower grid threshold satisfies it (minimal)', ok8)
ok9 = True
for sym in SYMS:
    ev = PROBE[sym]['events']; st = sorted((e['start'], e['L']) for e in ev); a_, b_ = BOUNDS['val']
    ok9 &= bool(len(ev) == 117 and all(sum(1 for e in ev if e['stratum'] == s) == 13 for s in range(9)) and all(st[j + 1][0] >= st[j][0] + st[j][1] + 60 for j in range(len(st) - 1)) and all(a_ <= e['start'] and e['start'] + e['L'] <= b_ for e in ev))
    diff = np.flatnonzero((PROBE[sym]['Z'] != Z1[sym]).any(axis=1)); span = np.zeros(N_BARS, bool)
    for e in ev: span[e['start']:e['start'] + e['L']] = True
    ok9 &= bool(span[diff].all() and not PROBE[sym]['neg'][span].any())
T(9, 'probe: 117 events (9 strata x 13) per symbol, spaced >= 60 bars, inside Layer 1 VALIDATION, perturbing only event bars', ok9)
stj = json.loads((STAGE / 'reports' / 'selection_table.json').read_text()); tab = pd.DataFrame(stj['selection_table']); vr = json.loads((STAGE / 'reports' / 'validation_results.json').read_text())['recommendation']
t_ok = tab[tab.gate_ok]; best_ = t_ok.probe.max(); tied_ = t_ok[t_ok.probe >= best_ - 0.02]; tied_ = tied_[tied_.simp_rank == tied_.simp_rank.min()].sort_values(['probe', 'scorer'], ascending=[False, True]); sel_re = tied_.iloc[0].scorer
dd = json.loads((STAGE / 'reports' / 'diagnosis_report.json').read_text())['validation_all_candidates']; rs = np.mean([dd[f'{s}|{sel_re}']['ops']['b2']['event_recall'] for s in SYMS]); rv = np.mean([dd[f'{s}|sb|cur|fmean']['ops']['b2']['event_recall'] for s in SYMS]); rec_re = 'sb|cur|fmean' if rs < rv else sel_re
T(10, 'selection and fallback recomputed from the saved probe table / validation results alone', sel_re == SELECTED == vr['selected_by_probe'] and rec_re == RECOMMENDED == vr['recommended'] and SELECTED in set(PT.index), (sel_re, SELECTED, rec_re, RECOMMENDED))
T(11, 'test split evaluated in exactly one unlocked pass, only for the declared final set, after validation results were saved', TEST_STATE['calls'] == len(SYMS) * len(DECL) and set(RES_TEST) == {f'{s}|{c}' for s in SYMS for c in DECL} and TS['val_saved'] < TS['test_unlocked'] < TS['test_evaluated'])
ok12 = True; ev_all = EV3
for sym in SYMS:
    for sc in DECL:
        for sp in ('val', 'test'):
            sdf = pd.read_parquet(STAGE / 'scores' / f'{SYMS[sym]}_{fname(sc)}_scores_l3.parquet')['score'].to_numpy(); R = (RES_VAL if sp == 'val' else RES_TEST)[f'{sym}|{sc}']
            for op, o in R['ops'].items():
                evs = ev_all[(ev_all.symbol == sym) & (ev_all.split == sp)]; det = 0
                for r in evs.itertuples():
                    hit = False
                    for i in range(int(r.start_idx), int(r.end_idx) + 1):
                        if sdf[i] > o['threshold']: hit = True; break
                    det += hit
                ok12 &= bool(det == o['events_detected'])
T(12, 'event-level recall independently recomputed (plain loop on stored scores) equals reported values, validation and test, every operating point', ok12)
if ARCH_RUN:
    fam0 = [f for f in ('ctx', 'dn') if ('BTCUSDT', f) in NEWM][0]; m2, rep2 = train_cae(fam0, 'BTCUSDT', SEEDS[0]); old = NEWM[('BTCUSDT', fam0)]; NEWM[('BTCUSDT', fam0)] = [m2]
    e2 = errs('BTCUSDT', fam0, Z1['BTCUSDT']).mean(axis=1); NEWM[('BTCUSDT', fam0)] = [old[0]]; e1 = errs('BTCUSDT', fam0, Z1['BTCUSDT']).mean(axis=1); NEWM[('BTCUSDT', fam0)] = old
    sp_ = stats.spearmanr(e1[100:], e2[100:])[0]; T(13, 'retraining the same seed reproduces the new model scores (Spearman >= 0.999)', sp_ >= 0.999, f'spearman {sp_:.5f}')
else: T(13, 'retraining check (not applicable: no new model was trained; v1 models are verified in T05)', True)
fin = all(np.isfinite(pd.read_parquet(STAGE / 'scores' / f'{SYMS[s]}_{fname(sc)}_scores_{src}.parquet').score.to_numpy()[100:]).all() for s in SYMS for sc in DECL for src in ('l1', 'l3'))
T(14, 'all stored scores finite after the warm-up', fin)
man = json.loads((STAGE / 'manifest.json').read_text())['files']; actual = {str(p.relative_to(STAGE)) for p in STAGE.rglob('*') if p.is_file() and p.name not in ('manifest.json', 'validation_report.json')}
T(15, 'manifest lists every file with correct sha256/size', set(man) == actual and all(sha256_file(STAGE / k) == v['sha256'] for k, v in man.items()), sorted(actual ^ set(man))[:5])
for p in STAGE.rglob('*.json'): json.loads(p.read_text())
for p in STAGE.rglob('*.parquet'): pq.read_metadata(p)
T(16, 'all JSON/parquet outputs readable', True)
ALL_PASS = all(r['pass'] for r in RES)
rv_ = lambda sc, R=None: float(np.mean([(R or RES_VAL)[f'{s}|{sc}']['ops']['b2']['event_recall'] for s in SYMS]))
IMPROVED = bool(RECOMMENDED != 'sb|cur|fmean' and R_SEL >= R_V1SB + 0.05); STRENGTH = 'weak' if rv_(RECOMMENDED) < 0.35 else ('moderate' if rv_(RECOMMENDED) < 0.6 else 'good')
if not ALL_PASS: VERDICT = 'BLOCKED'
elif IMPROVED: VERDICT = f'IMPROVEMENT VALIDATED on Layer 3 validation at matched burden; absolute detection {STRENGTH}; test exploratory'
elif RECOMMENDED == 'sb|cur|fmean': VERDICT = f'NO VALIDATED IMPROVEMENT over v1 single-bar; v1 single-bar RECOMMENDED (matched-burden recall {STRENGTH}); 15-bar model not recommended'
else: VERDICT = f'MARGINAL: recommended model not distinguishable from v1 single-bar (absolute detection {STRENGTH})'
print(f"\n{sum(r['pass'] for r in RES)}/{len(RES)} tests passed | {VERDICT}")
dump_json({'tests': RES, 'verdict': VERDICT, 'protocol_sha256': PROTOCOL_HASH, 'run_started_utc': RUN_STARTED}, STAGE / 'validation' / 'validation_report.json')
assert {n: pin_dir(BASE / n) for n in PIN} == PIN, 'frozen inputs changed during the run'
if ALL_PASS: os.rename(STAGE, OUT_FINAL); where = OUT_FINAL; print('PUBLISHED ->', where)
else: where = STAGE; print('NOT PUBLISHED; staging kept:', where)

def r2(x): return None if x is None else round(float(x), 3)
def tab_rows(R, op, scs):
    out = {}
    for sym in SYMS:
        for sc in scs:
            v = R[f'{sym}|{sc}']; o = v['ops'][op]; out[f'{sym[:3]}|{sc}'] = {'ROC': r2(v['roc_auc']), 'PR': r2(v['pr_auc']), 'chance': r2(v['pr_auc_chance']), 'recall': f"{o['events_detected']}/{v['n_events']} ({o['event_recall_ci95'][0]:.2f}-{o['event_recall_ci95'][1]:.2f})", 'delay': o['median_detection_delay_bars'], 'L1 ep/d': r2(v['layer1'][op]['episodes_per_day']), 'L3clean ep/d': r2(o['clean_negative_episodes_per_day']), 'fam': f"{o['families_covered']}/{o['families_total']}"}
    return pd.DataFrame(out).T
def fam_tab(R, op, scs): return pd.DataFrame({f'{sym[:3]}|{sc}': {f: f'{a}/{n}' for f, (a, n) in R[f'{sym}|{sc}']['ops'][op]['by_family'].items()} for sym in SYMS for sc in scs}).to_string()
print('\n==================== AUTOENCODER v2 — FINAL SHARE BLOCK ====================')
print('1. Output:', where, '| protocol sha256:', PROTOCOL_HASH[:16], '(frozen before any v2 score) | v1 protocol:', V1_HASH[:16])
print('2. Pipeline tests:', f"{sum(r['pass'] for r in RES)}/{len(RES)}", '| device', DEVICE, '| architecture stage ran:', ARCH_RUN, '| gate:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in GATE_INFO.items()})
print('3. Dilution diagnosis (Layer 3 VALIDATION, descriptive): top-position share of window error (uniform = 0.067):'); print(json.dumps({s: {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items() if k != 'threshold_over_median_genuine_score'} for s, d in DILUTION.items()}, indent=1))
print('   threshold / median genuine score (heavy-tail index):', {s: {k: round(v, 1) for k, v in d['threshold_over_median_genuine_score'].items()} for s, d in DILUTION.items()})
print('4. Models/scoring compared at matched burden (budget', PB, 'episodes/day). Probe = Layer 1 pseudo-anomaly recall; L3 val = event recall; gate = burden transfer:')
cmp_ = PT[['probe', 'short(1-3)', 'mid(4-12)', 'long(13-40)']].join(SELTAB.set_index('scorer')[['gate_ok']]); cmp_['L3 val recall (BTC)'] = [DIAG[f'BTCUSDT|{s}']['ops']['b2']['event_recall'] for s in cmp_.index]; cmp_['L3 val recall (ETH)'] = [DIAG[f'ETHUSDT|{s}']['ops']['b2']['event_recall'] for s in cmp_.index]; print(cmp_.round(3).to_string())
print('5. Selected by probe:', SELECTED, '| L3-val recall b2 selected', round(R_SEL, 3), 'vs v1 single-bar', round(R_V1SB, 3), '| fallback triggered:', FALLBACK, '| RECOMMENDED:', RECOMMENDED)
for nm, R in (('6. VALIDATION', RES_VAL), ('7. TEST (exploratory; v1 already examined this split)', RES_TEST)):
    print(nm, '@ matched budget 2 episodes/day (Layer 3 events; BTC/ETH):'); print(tab_rows(R, 'b2', DECL).to_string())
print('8. Event recall by scenario @ budget 2 (VALIDATION):'); print(fam_tab(RES_VAL, 'b2', DECL)); print('   (TEST, exploratory):'); print(fam_tab(RES_TEST, 'b2', DECL))
print('9. Alarm episodes/day on genuine Layer 1 (val | test) by budget:'); print(pd.DataFrame({f'{s[:3]}|{c}': {f'{op} val': RES_VAL[f'{s}|{c}']['layer1'][op]['episodes_per_day'] for op in RES_VAL[f'{s}|{c}']['layer1']} | {f'{op} test': RES_TEST[f'{s}|{c}']['layer1'][op]['episodes_per_day'] for op in RES_TEST[f'{s}|{c}']['layer1']} for s in SYMS for c in DECL}).T.round(2).to_string())
print('10. Comparison with v1 at its native q99.5 operating point (VALIDATION, then TEST):')
for nm, R in (('val', RES_VAL), ('test', RES_TEST)): print(' ', nm); print(tab_rows(R, 'v1_q99.5', [c for c in DECL if c in V1MAP]).to_string())
print('11. OOD behaviour @ budget 2 (recall on OOD-flagged events, TEST): ', {f'{s[:3]}|{c}': RES_TEST[f'{s}|{c}']['ops']['b2']['by_ood'] for s in SYMS for c in (RECOMMENDED,)})
print('12. Limitations: synthetic labels, no spoofing/layering; probe is generic and lacks cross-feature coherence; test already examined in v1 (exploratory); thresholds from ~13 validation days; scores uncalibrated; modest event counts; 3 seeds.')
print('13. FINAL VERDICT:', VERDICT)
print('==============================================================================')
if not ALL_PASS: raise AssertionError('validation failed; nothing published')

PASS T01 all required files exist
PASS T02 frozen sources unchanged (Layers 1, 2, 3 v1/v2, models_autoencoder_v1)
PASS T03 protocol intact (hash matches) and frozen BEFORE any score/probe/model; Layer 3 diagnosis and test only after selection
PASS T04 preprocessing fitted on Layer 1 TRAIN bars only (independent numpy recompute matches v2 and v1 files)
PASS T05 v1 models reload and reproduce the saved v1 Layer 1 and Layer 3 scores
PASS T06 all scorers are causal (changing bars after t leaves scores at <= t unchanged) and react to later changes
PASS T07 new-model training and early-stopping windows lie inside Layer 1 TRAIN and after the first legal bar
PASS T08 every budget threshold recomputed from Layer 1 VALIDATION: episodes/day <= budget and no lower grid threshold satisfies it (minimal)
PASS T09 probe: 117 events (9 strata x 13) per symbol, spaced >= 60 bars, inside Layer 1 VALIDATION, perturbing only event bars
PASS T10 selection and fallback recomputed from the saved probe table /